# LeetCode 1043 — Partition Array for Maximum Sum bằng Python

Link đề: https://leetcode.com/problems/partition-array-for-maximum-sum/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def maxSumAfterPartitioning(self, arr: List[int], k: int) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất** (tổng lớn nhất sau khi chia mảng thành các đoạn liên tiếp độ dài `≤ k` rồi thay mỗi phần tử bằng **giá trị lớn nhất của đoạn chứa nó**). Điểm cốt lõi: đoạn cuối có độ dài `L ≤ k`, đóng góp `max(đoạn) · L`, phần còn lại là cùng bài toán trên tiền tố ngắn hơn ⇒ **DP trên tiền tố**, `f[i] = max_{1 ≤ L ≤ min(k,i)} f[i-L] + L · max(arr[i-L..i-1])`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia mảng thành các đoạn liên tiếp mỗi đoạn dài **tối đa** `k`; mỗi đoạn được "nâng" mọi phần tử lên bằng phần tử lớn nhất của đoạn; tính tổng lớn nhất.
2. Nắm công cụ: DP trên tiền tố `f[i]`, cửa sổ lùi giữ `max` chạy, bộ nhớ đệm (memoization), DP xuôi/ngược.
3. Đi từ **vét cạn mọi cách chia `2^(n-1)`** → **đệ quy có nhớ** → **DP tiền tố** → **DP hậu tố** → **DP đẩy (push)**.
4. Hiểu mấu chốt: bài toán có **cấu trúc con tối ưu** (tối ưu của `n` phần tử = tối ưu của tiền tố ngắn hơn + một đoạn cuối) và **các bài con gối nhau** (`f[i]` được dùng lại bởi `k` trạng thái sau).
5. Tránh lỗi kinh điển: chia thành các khối **đúng** `k` phần tử, quên đặt lại `max` khi bắt đầu đoạn mới, nhân sai độ dài đoạn (lệch `1`), bỏ sót độ dài `k`, tham lam "mở rộng khi phần tử mới ≥ max".

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr`. Hãy chia `arr` thành các **đoạn liên tiếp** có độ dài **tối đa** `k`. Sau khi chia, **mỗi phần tử của đoạn được đổi thành giá trị lớn nhất của đoạn đó**. Trả về **tổng lớn nhất** của mảng sau khi chia. Dữ liệu được sinh sao cho đáp án nằm trong số nguyên `32` bit.

**Ví dụ 1:** `arr = [1,15,7,9,2,5,10]`, `k = 3` → `84` (mảng thành `[15,15,15,9,10,10,10]`).

**Ví dụ 2:** `arr = [1,4,1,5,7,3,6,1,9,9,3]`, `k = 4` → `83`.

**Ví dụ 3:** `arr = [1]`, `k = 1` → `1`.

**Ràng buộc:** `1 <= arr.length <= 500`; `0 <= arr[i] <= 10^9`; `1 <= k <= arr.length`.

In [1]:
import itertools
import random
import sys
import time
from functools import lru_cache
from typing import List

sys.setrecursionlimit(10000)

examples = [([1, 15, 7, 9, 2, 5, 10], 3, 84), ([1, 4, 1, 5, 7, 3, 6, 1, 9, 9, 3], 4, 83), ([1], 1, 1)]
for arr, k, want in examples:
    print(f"arr = {arr}, k = {k} -> {want}")

arr = [1, 15, 7, 9, 2, 5, 10], k = 3 -> 84
arr = [1, 4, 1, 5, 7, 3, 6, 1, 9, 9, 3], k = 4 -> 83
arr = [1], k = 1 -> 1


## 3. Kiến thức nền

### 3.1 Giá trị của một cách chia

Một cách chia là dãy các đoạn `[l_1, r_1], [l_2, r_2], …` phủ kín `arr`, mỗi đoạn dài `≤ k`. Giá trị của đoạn `[l, r]` là `(r - l + 1) · max(arr[l..r])`. Giá trị cách chia là tổng giá trị các đoạn. Ví dụ `[1,15,7 | 9,2,5 | 10]` có giá trị `3·15 + 3·9 + 1·10 = 82`, còn `[1,15,7 | 9 | 2,5,10]` có `45 + 9 + 30 = 84`.

In [2]:
def partition_value(arr, lengths):
    """Giá trị của cách chia theo danh sách độ dài các đoạn liên tiếp."""
    assert sum(lengths) == len(arr)
    total, i = 0, 0
    for L in lengths:
        total += L * max(arr[i:i + L])
        i += L
    return total


arr = [1, 15, 7, 9, 2, 5, 10]
print(partition_value(arr, [3, 3, 1]), partition_value(arr, [3, 1, 3]), partition_value(arr, [1] * 7))
assert partition_value(arr, [3, 1, 3]) == 84

82 84 49


### 3.2 Cấu trúc con tối ưu

Xét cách chia tối ưu của `arr[0..i-1]`. Đoạn cuối của nó là `arr[i-L..i-1]` với độ dài `L ∈ [1, min(k, i)]`. Phần còn lại là một cách chia của tiền tố `arr[0..i-L-1]` và **không ảnh hưởng** giá trị đoạn cuối. Vậy nếu phần còn lại không tối ưu, thay bằng cách chia tối ưu sẽ tốt hơn: **cấu trúc con tối ưu**. Đặt `f[i]` = đáp án cho `i` phần tử đầu (`f[0] = 0`):

`f[i] = max_{1 ≤ L ≤ min(k, i)} ( f[i-L] + L · max(arr[i-L], …, arr[i-1]) )`.

### 3.3 Duy trì `max` khi lùi

Khi `L` tăng thêm `1`, đoạn mở rộng sang trái thêm `arr[i-L]` nên `max` mới = `max(max cũ, arr[i-L])`: cập nhật `O(1)`. Nhờ vậy duyệt tất cả `L ≤ k` tốn `O(k)` cho mỗi `i`.

## 4. Ý tưởng nền tảng / trực giác

Hình dung ta chốt **đoạn cuối** của mảng: nó có thể dài từ `1` đến `k`. Với mỗi lựa chọn độ dài, đoạn cuối đóng góp `độ dài × max`, phần còn lại là **cùng bài toán** trên mảng ngắn hơn đã giải trước đó. Ta thử hết các độ dài rồi lấy tốt nhất. Mỗi kết quả `f[j]` được dùng lại nhiều lần nên lưu bảng.

Vì sao không tham lam? Mở rộng một đoạn để "ăn" một phần tử lớn có thể kéo theo các phần tử nhỏ đứng cạnh lên giá trị lớn (có lợi), nhưng cũng có thể cướp mất cơ hội để phần tử lớn kế tiếp tạo đoạn riêng với nhiều phần tử hơn. Quyết định cục bộ không đủ ⇒ cần DP.

## 5. Brute force — thử mọi cách chia

**Ý tưởng:** giữa `n` phần tử có `n - 1` khe; mỗi khe có thể **cắt** hoặc **không cắt**: `2^(n-1)` cách chia. Với mỗi cách (mặt nạ bit), tách thành các đoạn; nếu có đoạn dài hơn `k` thì bỏ; ngược lại tính giá trị và lấy lớn nhất. Đúng theo định nghĩa, dùng làm **oracle** cho `n ≤ 14`.

In [3]:
def max_sum_brute(arr, k):
    n = len(arr)
    best = 0
    for mask in range(1 << (n - 1)):                       # bit j = 1: cắt giữa vị trí j và j + 1
        total, start, ok = 0, 0, True
        for j in range(n):
            if j == n - 1 or (mask >> j) & 1:              # kết thúc một đoạn tại j
                L = j - start + 1
                if L > k:
                    ok = False
                    break
                total += L * max(arr[start:j + 1])
                start = j + 1
        if ok and total > best:
            best = total
    return best


for arr, k, want in examples[:1] + examples[2:]:
    assert max_sum_brute(arr, k) == want
print([max_sum_brute(a, k) for a, k, _ in examples if len(a) <= 14])

[84, 83, 1]


In [4]:
# Code nộp LeetCode cho Brute force (đúng nhưng chỉ chạy nổi n <= ~20; n = 500 sẽ TLE)
class SolutionBrute:
    def maxSumAfterPartitioning(self, arr: List[int], k: int) -> int:
        n = len(arr)
        best = 0
        for mask in range(1 << (n - 1)):
            total, start, ok = 0, 0, True
            for j in range(n):
                if j == n - 1 or (mask >> j) & 1:
                    L = j - start + 1
                    if L > k:
                        ok = False
                        break
                    total += L * max(arr[start:j + 1])
                    start = j + 1
            if ok and total > best:
                best = total
        return best

**Độ phức tạp:** `O(2^n · n)` thời gian, `O(1)` bộ nhớ phụ.

**Nút thắt:** hai cách chia khác nhau thường chia sẻ cùng một tiền tố tối ưu — ta tính đi tính lại.

## 6. Bước đột phá

Chốt **đoạn cuối** (độ dài `L ≤ k`) và dùng đáp án tối ưu của tiền tố còn lại: `f[i] = max_L f[i-L] + L · max(đoạn cuối)`. Có `n` trạng thái, mỗi trạng thái thử `≤ k` độ dài ⇒ `O(nk)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** đệ quy có nhớ (top-down), **Cách 3** DP tiền tố (đáp án chính), **Cách 4** DP hậu tố, **Cách 5** DP đẩy (push).

### Cách 2 — Đệ quy có nhớ (top-down)

**Giải thích:** `go(i)` = đáp án tối ưu cho **hậu tố** `arr[i:]`. Ta chọn độ dài đoạn **đầu** `L ∈ [1, min(k, n - i)]`: giá trị `L · max(arr[i:i+L]) + go(i + L)`, `go(n) = 0`. Có nhớ bằng `lru_cache`.

In [5]:
def max_sum_topdown(arr, k):
    n = len(arr)

    @lru_cache(maxsize=None)
    def go(i):
        if i == n:
            return 0
        best, mx = 0, 0
        for L in range(1, min(k, n - i) + 1):
            mx = max(mx, arr[i + L - 1])                   # max của arr[i : i + L], cập nhật dần
            best = max(best, L * mx + go(i + L))
        return best

    return go(0)


for arr, k, want in examples:
    assert max_sum_topdown(arr, k) == want
print([max_sum_topdown(a, k) for a, k, _ in examples])

[84, 83, 1]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionTopDown:
    def maxSumAfterPartitioning(self, arr: List[int], k: int) -> int:
        n = len(arr)

        @lru_cache(maxsize=None)
        def go(i):
            if i == n:
                return 0
            best, mx = 0, 0
            for L in range(1, min(k, n - i) + 1):
                mx = max(mx, arr[i + L - 1])
                best = max(best, L * mx + go(i + L))
            return best

        return go(0)

**Độ phức tạp:** `O(nk)` thời gian (`n` trạng thái × `≤ k` lựa chọn), `O(n)` bộ nhớ (bảng nhớ + ngăn xếp đệ quy sâu tới `n/1 = 500` lớp).

### Cách 3 — DP tiền tố (đáp án chính)

**Ý tưởng:** `f[0] = 0`; với `i = 1..n`, duyệt `L = 1..min(k, i)`, giữ `mx` = max của `arr[i-L..i-1]`, và `f[i] = max(f[i-L] + L·mx)`. Đáp án `f[n]`.

In [7]:
def max_sum_prefix(arr, k):
    n = len(arr)
    f = [0] * (n + 1)
    for i in range(1, n + 1):
        mx = 0
        for L in range(1, min(k, i) + 1):
            mx = max(mx, arr[i - L])                       # mở rộng đoạn sang trái
            f[i] = max(f[i], f[i - L] + L * mx)
    return f[n]


for arr, k, want in examples:
    assert max_sum_prefix(arr, k) == want
print([max_sum_prefix(a, k) for a, k, _ in examples])

[84, 83, 1]


In [8]:
# Code nộp LeetCode cho Cách 3
class SolutionPrefix:
    def maxSumAfterPartitioning(self, arr: List[int], k: int) -> int:
        n = len(arr)
        f = [0] * (n + 1)
        for i in range(1, n + 1):
            mx = 0
            for L in range(1, min(k, i) + 1):
                mx = max(mx, arr[i - L])
                f[i] = max(f[i], f[i - L] + L * mx)
        return f[n]

**Độ phức tạp:** `O(nk)` thời gian, `O(n)` bộ nhớ.

### Cách 4 — DP hậu tố (bottom-up ngược)

**Ý tưởng:** bản bottom-up của Cách 2: `g[i]` = đáp án cho hậu tố `arr[i:]`; `g[n] = 0`; với `i = n-1..0`, thử độ dài đoạn đầu `L`. Đáp án `g[0]`. Kết quả bằng Cách 3 (đối xứng gương).

In [9]:
def max_sum_suffix(arr, k):
    n = len(arr)
    g = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        mx = 0
        for L in range(1, min(k, n - i) + 1):
            mx = max(mx, arr[i + L - 1])
            g[i] = max(g[i], L * mx + g[i + L])
    return g[0]


for arr, k, want in examples:
    assert max_sum_suffix(arr, k) == want
print([max_sum_suffix(a, k) for a, k, _ in examples])

[84, 83, 1]


In [10]:
# Code nộp LeetCode cho Cách 4
class SolutionSuffix:
    def maxSumAfterPartitioning(self, arr: List[int], k: int) -> int:
        n = len(arr)
        g = [0] * (n + 1)
        for i in range(n - 1, -1, -1):
            mx = 0
            for L in range(1, min(k, n - i) + 1):
                mx = max(mx, arr[i + L - 1])
                g[i] = max(g[i], L * mx + g[i + L])
        return g[0]

**Độ phức tạp:** `O(nk)` thời gian, `O(n)` bộ nhớ.

### Cách 5 — DP đẩy (push / forward relaxation)

**Ý tưởng:** thay vì "kéo" `f[i]` từ `k` trạng thái trước, **từ `f[i]` đã chốt**, "đẩy" tới `f[i+L]` cho `L = 1..k`: `f[i+L] = max(f[i+L], f[i] + L · max(arr[i..i+L-1]))`. Cùng đồ thị trạng thái, chỉ khác hướng duyệt.

In [11]:
def max_sum_push(arr, k):
    n = len(arr)
    f = [-1] * (n + 1)
    f[0] = 0
    for i in range(n):                                     # f[i] đã chốt (mọi đường đến i đi qua chỉ số < i)
        mx = 0
        for L in range(1, min(k, n - i) + 1):
            mx = max(mx, arr[i + L - 1])
            f[i + L] = max(f[i + L], f[i] + L * mx)
    return f[n]


for arr, k, want in examples:
    assert max_sum_push(arr, k) == want
print([max_sum_push(a, k) for a, k, _ in examples])

[84, 83, 1]


In [12]:
# Code nộp LeetCode cho Cách 5
class SolutionPush:
    def maxSumAfterPartitioning(self, arr: List[int], k: int) -> int:
        n = len(arr)
        f = [-1] * (n + 1)
        f[0] = 0
        for i in range(n):
            mx = 0
            for L in range(1, min(k, n - i) + 1):
                mx = max(mx, arr[i + L - 1])
                f[i + L] = max(f[i + L], f[i] + L * mx)
        return f[n]

**Độ phức tạp:** `O(nk)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace **Cách 3** với `arr = [1,15,7,9,2,5,10]`, `k = 3`. Cột `L` cho biết độ dài đoạn cuối thử, `mx` là max của đoạn đó, giá trị `= f[i-L] + L·mx`.

| `i` | các lựa chọn `(L, mx, f[i-L] + L·mx)` | `f[i]` |
|---|---|---|
| 0 | — | 0 |
| 1 | (1,1,1) | 1 |
| 2 | (1,15,16), (2,15,30) | 30 |
| 3 | (1,7,37), (2,15,31), (3,15,45) | 45 |
| 4 | (1,9,54), (2,9,48), (3,15,46) | 54 |
| 5 | (1,2,56), (2,9,63), (3,9,57) | 63 |
| 6 | (1,5,68), (2,5,64), (3,9,72) | 72 |
| 7 | (1,10,82), (2,10,83), (3,10,84) | **84** |

Hàm trace in từng bước và kiểm tra bảng `f`:

In [13]:
def trace_prefix(arr, k):
    n = len(arr)
    f = [0] * (n + 1)
    print(f"arr = {arr}, k = {k}")
    for i in range(1, n + 1):
        mx, opts = 0, []
        for L in range(1, min(k, i) + 1):
            mx = max(mx, arr[i - L])
            opts.append((L, mx, f[i - L] + L * mx))
            f[i] = max(f[i], f[i - L] + L * mx)
        print(f"  i = {i}: đoạn cuối arr[{i}-L..{i - 1}], các lựa chọn (L, mx, giá trị) = {opts} -> f[{i}] = {f[i]}")
    return f


f = trace_prefix([1, 15, 7, 9, 2, 5, 10], 3)
assert f == [0, 1, 30, 45, 54, 63, 72, 84]

arr = [1, 15, 7, 9, 2, 5, 10], k = 3
  i = 1: đoạn cuối arr[1-L..0], các lựa chọn (L, mx, giá trị) = [(1, 1, 1)] -> f[1] = 1
  i = 2: đoạn cuối arr[2-L..1], các lựa chọn (L, mx, giá trị) = [(1, 15, 16), (2, 15, 30)] -> f[2] = 30
  i = 3: đoạn cuối arr[3-L..2], các lựa chọn (L, mx, giá trị) = [(1, 7, 37), (2, 15, 31), (3, 15, 45)] -> f[3] = 45
  i = 4: đoạn cuối arr[4-L..3], các lựa chọn (L, mx, giá trị) = [(1, 9, 54), (2, 9, 48), (3, 15, 46)] -> f[4] = 54
  i = 5: đoạn cuối arr[5-L..4], các lựa chọn (L, mx, giá trị) = [(1, 2, 56), (2, 9, 63), (3, 9, 57)] -> f[5] = 63
  i = 6: đoạn cuối arr[6-L..5], các lựa chọn (L, mx, giá trị) = [(1, 5, 68), (2, 5, 64), (3, 9, 72)] -> f[6] = 72
  i = 7: đoạn cuối arr[7-L..6], các lựa chọn (L, mx, giá trị) = [(1, 10, 82), (2, 10, 83), (3, 10, 84)] -> f[7] = 84


**Khôi phục cách chia:** lưu lại `L` tốt nhất ở mỗi `i` rồi lần ngược từ `n`: `i = 7` chọn `L = 3` (đoạn `[2,5,10]`), `i = 4` chọn `L = 1` (đoạn `[9]`), `i = 3` chọn `L = 3` (đoạn `[1,15,7]`) ⇒ chia `[1,15,7 | 9 | 2,5,10]`, mảng thành `[15,15,15,9,10,10,10]`, tổng `84`.

In [14]:
def best_partition(arr, k):
    n = len(arr)
    f = [0] * (n + 1)
    choice = [0] * (n + 1)
    for i in range(1, n + 1):
        mx = 0
        f[i] = -1                                          # -1: chưa có lựa chọn nào (tránh kẹt khi mọi giá trị bằng 0)
        for L in range(1, min(k, i) + 1):
            mx = max(mx, arr[i - L])
            if f[i - L] + L * mx > f[i]:
                f[i], choice[i] = f[i - L] + L * mx, L
    lengths, i = [], n
    while i > 0:
        lengths.append(choice[i])
        i -= choice[i]
    return f[n], lengths[::-1]


val, lens = best_partition([1, 15, 7, 9, 2, 5, 10], 3)
print(val, lens)
assert val == 84 and lens == [3, 1, 3] and partition_value([1, 15, 7, 9, 2, 5, 10], lens) == 84

84 [3, 1, 3]


## 9. Code nộp LeetCode

Chọn **Cách 3** (DP tiền tố). Giữ nguyên `class Solution`, `maxSumAfterPartitioning(self, arr, k)`:

In [15]:
class Solution:
    def maxSumAfterPartitioning(self, arr: List[int], k: int) -> int:
        n = len(arr)
        f = [0] * (n + 1)                           # f[i] = đáp án cho i phần tử đầu
        for i in range(1, n + 1):
            mx = 0
            for L in range(1, min(k, i) + 1):       # độ dài đoạn cuối: 1..min(k, i)
                mx = max(mx, arr[i - L])            # max của arr[i-L .. i-1], mở rộng sang trái
                f[i] = max(f[i], f[i - L] + L * mx)
        return f[n]


print([Solution().maxSumAfterPartitioning(a, k) for a, k, _ in examples])
assert all(Solution().maxSumAfterPartitioning(a, k) == w for a, k, w in examples)

[84, 83, 1]


## 10. Vì sao đúng

**(a) Cấu trúc con tối ưu.** Gọi `OPT(i)` là giá trị lớn nhất của một cách chia hợp lệ của `arr[0..i-1]`. Lấy một cách chia tối ưu; đoạn cuối có độ dài `L ∈ [1, min(k, i)]`. Giá trị = (giá trị đoạn cuối, chỉ phụ thuộc `arr[i-L..i-1]`) + (giá trị phần còn lại, là một cách chia hợp lệ của `arr[0..i-L-1]`). Nếu phần còn lại chưa tối ưu thì thay bằng cách chia tối ưu của tiền tố đó, giá trị tổng tăng nhưng vẫn hợp lệ — mâu thuẫn. Vậy `OPT(i) = max_L ( OPT(i-L) + L · max(arr[i-L..i-1]) )`.

**(b) Hai chiều của bất đẳng thức.** (≥) Với mỗi `L`, ghép cách chia tối ưu của tiền tố `i-L` với đoạn cuối `L` cho một cách chia hợp lệ của `i` phần tử nên `OPT(i) ≥` mỗi số hạng. (≤) Theo (a) tồn tại `L` để đạt bằng. Do đó `f[i] = OPT(i)` bằng quy nạp theo `i`, `f[0] = 0` là cơ sở (mảng rỗng).

**(c) Duy trì `max`.** Khi `L` tăng lên `L + 1`, đoạn là `arr[i-L-1..i-1]` nên `max` mới = `max(max cũ, arr[i-L-1])`; vòng lặp cập nhật `mx = max(mx, arr[i - L])` với `L` mới. Đúng.

**(d) Các cách còn lại.** Cách 2 và 4 áp dụng (a) với **đoạn đầu** (đối xứng gương), Cách 5 là cùng đồ thị trạng thái `i → i + L` nhưng thực hiện "đẩy"; thứ tự `i` tăng đảm bảo `f[i]` đã chốt khi đẩy. Cách 1 là định nghĩa. Ta kiểm bằng máy: mọi cách khớp brute trên toàn bộ mảng nhỏ và ngẫu nhiên (mục 12).

**(e) Giá trị lớn nhất theo biên.** `f[n] ≤ n · max(arr)` (mỗi phần tử không vượt `max` toàn cục) và `f[n] ≥ sum(arr)` (chia thành các đoạn dài `1`). Với `k = 1` đáp án là `sum(arr)`; với `k = n` là `n · max(arr)`. Các biên này được dùng làm test tính chất.

**(f) Bất biến dịch/đảo.** Cộng hằng `c ≥ 0` vào mọi phần tử làm `max` mỗi đoạn tăng đúng `c`, tổng tăng `n·c` với **mọi** cách chia, nên đáp án tăng `n·c`. Nhân mọi phần tử với `c ≥ 0` nhân đáp án với `c`. Đảo mảng cho cùng đáp án (tập cách chia đối xứng).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(1)` | oracle `n ≤ 14` |
| 2. Top-down | `O(nk)` | `O(n)` | đệ quy sâu `n` |
| 3. Prefix DP | `O(nk)` | `O(n)` | **đáp án chính** |
| 4. Suffix DP | `O(nk)` | `O(n)` | đối xứng gương |
| 5. Push DP | `O(nk)` | `O(n)` | cùng đồ thị, khác hướng |

Với `n = 500`: `nk ≤ 250 000` phép tính, rất nhanh. Đo thực tế (`n = 500`, `k = 250`):

In [16]:
def bench(fn, *args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(1043)
big = [rng.randint(0, 10 ** 9) for _ in range(500)]
for name, fn in [("topdown", max_sum_topdown), ("prefix", max_sum_prefix), ("suffix", max_sum_suffix), ("push", max_sum_push)]:
    print(f"{name:8s}: {bench(fn, big, 250) * 1000:7.1f} ms -> {fn(big, 250)}")

topdown :    26.9 ms -> 496136570793


prefix  :    16.5 ms -> 496136570793


suffix  :    12.1 ms -> 496136570793
push    :    13.8 ms -> 496136570793


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ cho lỗi hay gặp, (D) vét cạn mọi mảng nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "topdown": max_sum_topdown,
    "prefix": max_sum_prefix,
    "suffix": max_sum_suffix,
    "push": max_sum_push,
    "Solution": lambda a, k: Solution().maxSumAfterPartitioning(a, k),
    "SolutionTopDown": lambda a, k: SolutionTopDown().maxSumAfterPartitioning(a, k),
    "SolutionPrefix": lambda a, k: SolutionPrefix().maxSumAfterPartitioning(a, k),
    "SolutionSuffix": lambda a, k: SolutionSuffix().maxSumAfterPartitioning(a, k),
    "SolutionPush": lambda a, k: SolutionPush().maxSumAfterPartitioning(a, k),
}
BRUTE = {"brute": max_sum_brute, "SolutionBrute": lambda a, k: SolutionBrute().maxSumAfterPartitioning(a, k)}


def check(arr, k, want=None, brute=False):
    if want is None:
        want = max_sum_brute(arr, k) if brute else Solution().maxSumAfterPartitioning(arr, k)
    impls = dict(IMPLS)
    if brute:
        impls.update(BRUTE)
    for name, fn in impls.items():
        got = fn(list(arr), k)
        assert got == want, (name, arr, k, got, want)
    return want


for arr, k, want in examples:
    check(arr, k, want, brute=len(arr) <= 14)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "DP +", len(BRUTE), "vét cạn")

ví dụ đề: mọi cài đặt đúng; 9 DP + 2 vét cạn


### B. Edge case

- `n = 1` (`k = 1`): đáp án là chính phần tử.
- `k = 1`: đáp án là `sum(arr)` (không thể gộp).
- `k = n`: đáp án là `n · max(arr)`.
- Toàn số `0`; toàn số bằng nhau; mảng tăng dần; giảm dần; một đỉnh nhọn ở giữa.
- `k = 2` với hai phần tử bằng nhau / khác nhau.
- Số rất lớn `10^9` (đáp án vượt `32` bit khi `n` lớn — Python không tràn; đề đảm bảo test thật vừa `32` bit).

In [18]:
edge = [
    ([5], 1, 5), ([0], 1, 0), ([0, 0, 0, 0], 2, 0), ([7, 7, 7, 7, 7], 3, 35),
    ([1, 2, 3, 4, 5], 1, 15), ([1, 2, 3, 4, 5], 5, 25), ([5, 4, 3, 2, 1], 5, 25),
    ([1, 2, 3, 4, 5], 2, None), ([5, 4, 3, 2, 1], 2, None), ([1, 9, 1], 2, None), ([1, 9, 1], 3, 27),
    ([3, 1], 2, 6), ([1, 3], 2, 6), ([2, 2], 2, 4),
    ([10 ** 9, 0, 10 ** 9], 3, 3 * 10 ** 9), ([10 ** 9] * 8, 3, 8 * 10 ** 9),
]
for arr, k, want in edge:
    got = check(arr, k, want, brute=True)
    print(f"  arr = {arr}, k = {k} -> {got}")
print(len(edge), "edge case: mọi cài đặt (kể cả vét cạn) khớp")

  arr = [5], k = 1 -> 5
  arr = [0], k = 1 -> 0
  arr = [0, 0, 0, 0], k = 2 -> 0
  arr = [7, 7, 7, 7, 7], k = 3 -> 35
  arr = [1, 2, 3, 4, 5], k = 1 -> 15
  arr = [1, 2, 3, 4, 5], k = 5 -> 25
  arr = [5, 4, 3, 2, 1], k = 5 -> 25
  arr = [1, 2, 3, 4, 5], k = 2 -> 17
  arr = [5, 4, 3, 2, 1], k = 2 -> 17
  arr = [1, 9, 1], k = 2 -> 19
  arr = [1, 9, 1], k = 3 -> 27
  arr = [3, 1], k = 2 -> 6
  arr = [1, 3], k = 2 -> 6
  arr = [2, 2], k = 2 -> 4
  arr = [1000000000, 0, 1000000000], k = 3 -> 3000000000
  arr = [1000000000, 1000000000, 1000000000, 1000000000, 1000000000, 1000000000, 1000000000, 1000000000], k = 3 -> 8000000000
16 edge case: mọi cài đặt (kể cả vét cạn) khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_fixed_blocks(arr, k):                            # chia thành các khối đúng k phần tử (khối cuối ngắn hơn)
    return sum(len(arr[i:i + k]) * max(arr[i:i + k]) for i in range(0, len(arr), k))


def bug_no_reset_max(arr, k):                            # max lấy trên cả tiền tố, không chỉ đoạn cuối
    n = len(arr)
    f = [0] * (n + 1)
    for i in range(1, n + 1):
        for L in range(1, min(k, i) + 1):
            f[i] = max(f[i], f[i - L] + L * max(arr[:i]))
    return f[n]


def bug_len_off_by_one(arr, k):                          # nhân (L - 1) thay vì L
    n = len(arr)
    f = [0] * (n + 1)
    for i in range(1, n + 1):
        mx = 0
        for L in range(1, min(k, i) + 1):
            mx = max(mx, arr[i - L])
            f[i] = max(f[i], f[i - L] + (L - 1) * mx)
    return f[n]


def bug_miss_length_k(arr, k):                           # bỏ sót độ dài k (range thiếu +1)
    n = len(arr)
    f = [0] * (n + 1)
    for i in range(1, n + 1):
        mx = 0
        for L in range(1, min(k, i)):
            mx = max(mx, arr[i - L])
            f[i] = max(f[i], f[i - L] + L * mx)
        if k == 1 or i == 1:
            f[i] = max(f[i], f[i - 1] + arr[i - 1])
    return f[n]


def bug_greedy_extend(arr, k):                           # tham lam: mở rộng đoạn khi phần tử mới >= max hiện tại
    total, i, n = 0, 0, len(arr)
    while i < n:
        mx, L = arr[i], 1
        while i + L < n and L < k and arr[i + L] >= mx:
            mx = arr[i + L]
            L += 1
        total += mx * L
        i += L
    return total


def bug_stale_max(arr, k):                               # đặt mx một lần cho cả i, không cập nhật khi mở rộng
    n = len(arr)
    f = [0] * (n + 1)
    for i in range(1, n + 1):
        mx = arr[i - 1]
        for L in range(1, min(k, i) + 1):
            f[i] = max(f[i], f[i - L] + L * mx)
    return f[n]


BUGS = [bug_fixed_blocks, bug_no_reset_max, bug_len_off_by_one, bug_miss_length_k, bug_greedy_extend, bug_stale_max]
pool = [(list(a), k) for n in range(1, 7) for a in itertools.product(range(4), repeat=n) for k in range(1, n + 1)]
for bug in BUGS:
    found = None
    for a, k in pool:
        if bug(a, k) != max_sum_brute(a, k):
            found = (a, k)
            break
    assert found, bug.__name__
    a, k = found
    print(f"{bug.__name__:20s} sai trên arr = {a}, k = {k}: ra {bug(a, k)}, đúng {max_sum_brute(a, k)}")

bug_fixed_blocks     sai trên arr = [0, 0, 1], k = 2: ra 1, đúng 2
bug_no_reset_max     sai trên arr = [1, 0], k = 1: ra 2, đúng 1
bug_len_off_by_one   sai trên arr = [1], k = 1: ra 0, đúng 1
bug_miss_length_k    sai trên arr = [0, 1], k = 2: ra 1, đúng 2
bug_greedy_extend    sai trên arr = [1, 0], k = 2: ra 1, đúng 2
bug_stale_max        sai trên arr = [1, 0], k = 2: ra 1, đúng 2


### D. Vét cạn mọi mảng nhỏ

Mọi mảng độ dài `n = 1..6` với giá trị `0..3` và mọi `k = 1..n`, cộng mọi mảng `n = 7` với giá trị `0..2`: so với brute `2^(n-1)`. Mọi cài đặt DP và các lớp nộp phải khớp.

In [20]:
t0 = time.perf_counter()
cnt = 0
for n, vmax in [(1, 3), (2, 3), (3, 3), (4, 3), (5, 3), (6, 3), (7, 2)]:
    for tup in itertools.product(range(vmax + 1), repeat=n):
        a = list(tup)
        for k in range(1, n + 1):
            want = max_sum_brute(a, k)
            for name, fn in IMPLS.items():
                assert fn(a, k) == want, (name, a, k)
            cnt += 1
print(cnt, "cặp (mảng, k):", len(IMPLS), "cài đặt DP khớp vét cạn;", round(time.perf_counter() - t0, 1), "s")

46257 cặp (mảng, k): 9 cài đặt DP khớp vét cạn; 3.5 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) `n ≤ 14`, giá trị nhỏ và lớn: mọi cài đặt khớp brute; (2) `n ≤ 500`: các DP khớp nhau; (3) **biên**: `sum(arr) ≤ ans ≤ n·max`; `k = 1 → sum`; `k = n → n·max`; (4) **đơn điệu theo `k`**: `ans(k) ≤ ans(k+1)`; (5) **dịch**: `ans(arr + c) = ans(arr) + n·c`; (6) **nhân**: `ans(c·arr) = c·ans(arr)`; (7) **đảo**: `ans(reversed) = ans`; (8) khôi phục cách chia: `partition_value(lengths) = ans` và mọi đoạn `≤ k`.

In [21]:
rng = random.Random(20240143)
t0 = time.perf_counter()
for it in range(500):
    n = rng.randint(1, 14)
    a = [rng.randint(0, rng.choice([5, 30, 10 ** 9])) for _ in range(n)]
    k = rng.randint(1, n)
    check(a, k, brute=True)                                                          # (1)
print("(1) 500 mảng n <= 14 khớp vét cạn:", round(time.perf_counter() - t0, 1), "s")

t0 = time.perf_counter()
for it in range(150):
    n = rng.randint(1, 500)
    a = [rng.randint(0, rng.choice([3, 100, 10 ** 9])) for _ in range(n)]
    k = rng.randint(1, min(n, rng.choice([2, 5, 30, 500])))
    ans = check(a, k)                                                                 # (2)
    assert sum(a) <= ans <= n * max(a)                                                # (3)
    assert Solution().maxSumAfterPartitioning(a, 1) == sum(a)
    assert Solution().maxSumAfterPartitioning(a, n) == n * max(a)
    if k < n:
        assert ans <= Solution().maxSumAfterPartitioning(a, k + 1)                    # (4)
    c = rng.randint(0, 1000)
    assert Solution().maxSumAfterPartitioning([x + c for x in a], k) == ans + n * c   # (5)
    assert Solution().maxSumAfterPartitioning([x * c for x in a], k) == ans * c       # (6)
    assert Solution().maxSumAfterPartitioning(a[::-1], k) == ans                      # (7)
    val, lens = best_partition(a, k)
    assert val == ans and max(lens) <= k and sum(lens) == n and partition_value(a, lens) == ans   # (8)
print("(2)-(8) 150 mảng n <= 500: các tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

(1) 500 mảng n <= 14 khớp vét cạn: 3.5 s


(2)-(8) 150 mảng n <= 500: các tính chất đúng; 3.6 s


### F. Giới hạn

`n = 500`: mọi `k` đặc biệt (`1`, `2`, `n`, `n/2`), giá trị tối đa `10^9`, mảng hằng, tăng dần, giảm dần, răng cưa. Kiểm so với công thức đóng khi có (`k = 1`, `k = n`, mảng hằng) và đối chiếu các DP.

In [22]:
t0 = time.perf_counter()
n = 500
rg = random.Random(7)
cases = {
    "hằng 1e9": [10 ** 9] * n,
    "tăng dần": list(range(n)),
    "giảm dần": list(range(n, 0, -1)),
    "răng cưa": [10 ** 9 if i % 7 == 0 else i % 5 for i in range(n)],
    "ngẫu nhiên": [rg.randint(0, 10 ** 9) for _ in range(n)],
    "toàn 0": [0] * n,
}
for name, a in cases.items():
    for k in (1, 2, 7, 250, n):
        ans = check(a, k)
        if k == 1:
            assert ans == sum(a)
        if k == n:
            assert ans == n * max(a)
    print(f"  {name:12s}: k = 250 -> {Solution().maxSumAfterPartitioning(a, 250)}")
assert Solution().maxSumAfterPartitioning(cases["hằng 1e9"], 3) == 500 * 10 ** 9
print("n = 500, 5 giá trị k × 6 mảng:", round(time.perf_counter() - t0, 1), "s")

  hằng 1e9    : k = 250 -> 500000000000


  tăng dần    : k = 250 -> 187000


  giảm dần    : k = 250 -> 187500


  răng cưa    : k = 250 -> 500000000000


  ngẫu nhiên  : k = 250 -> 494233285250


  toàn 0      : k = 250 -> 0
n = 500, 5 giá trị k × 6 mảng: 2.0 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chia thành các khối đúng `k` phần tử

In [23]:
a, k = [1, 15, 7, 9, 2, 5, 10], 3
print("bug:", bug_fixed_blocks(a, k), "| đúng:", Solution().maxSumAfterPartitioning(a, k))
assert bug_fixed_blocks(a, k) == 82 and Solution().maxSumAfterPartitioning(a, k) == 84

bug: 82 | đúng: 84


**Vì sao sai:** đề cho phép đoạn dài **tối đa** `k`, không bắt buộc `k`. Chia cứng `[1,15,7 | 9,2,5 | 10]` cho `82 < 84`.

### Lỗi 2: `max` lấy trên cả tiền tố (không đặt lại theo đoạn)

In [24]:
a, k = next((a, k) for a, k in pool if bug_no_reset_max(a, k) != max_sum_brute(a, k))
print("phản ví dụ:", a, k, "| bug:", bug_no_reset_max(a, k), "| đúng:", max_sum_brute(a, k))
assert bug_no_reset_max(a, k) > max_sum_brute(a, k)

phản ví dụ: [1, 0] 1 | bug: 2 | đúng: 1


**Vì sao sai:** giá trị đoạn là `L · max` **của đoạn đó**, không phải max của cả tiền tố. Nếu lấy max toàn tiền tố thì đoạn cuối "mượn" phần tử lớn ở xa ngoài đoạn và cho kết quả **quá lớn**.

### Lỗi 3: nhân sai độ dài đoạn (lệch `1`)

In [25]:
a, k = [3], 1
print("bug:", bug_len_off_by_one(a, k), "| đúng:", Solution().maxSumAfterPartitioning(a, k))
assert bug_len_off_by_one(a, k) == 0 and Solution().maxSumAfterPartitioning(a, k) == 3

bug: 0 | đúng: 3


**Vì sao sai:** đoạn `arr[i-L..i-1]` có đúng `L` phần tử; nhân `(L - 1)` làm mất một phần tử.

### Lỗi 4: bỏ sót độ dài `k` do `range` thiếu `+1`

In [26]:
a, k = next((a, k) for a, k in pool if bug_miss_length_k(a, k) != max_sum_brute(a, k))
print("phản ví dụ:", a, k, "| bug:", bug_miss_length_k(a, k), "| đúng:", max_sum_brute(a, k))
assert bug_miss_length_k(a, k) != max_sum_brute(a, k)

phản ví dụ: [0, 1] 2 | bug: 1 | đúng: 2


**Vì sao sai:** `range(1, min(k, i))` dừng ở `min(k, i) - 1`, bỏ mất độ dài `k`. **Đúng:** `range(1, min(k, i) + 1)`.

### Lỗi 5: tham lam "mở rộng khi phần tử mới ≥ max"

In [27]:
a, k = next((a, k) for a, k in pool if bug_greedy_extend(a, k) != max_sum_brute(a, k))
print("phản ví dụ:", a, k, "| bug:", bug_greedy_extend(a, k), "| đúng:", max_sum_brute(a, k))
assert bug_greedy_extend(a, k) != max_sum_brute(a, k)

phản ví dụ: [1, 0] 2 | bug: 1 | đúng: 2


**Vì sao sai:** tham lam bỏ qua khả năng gộp phần tử nhỏ đứng **trước** phần tử lớn vào cùng đoạn (có lợi), hay không gộp để dành chỗ cho đoạn sau. Cần so sánh **mọi** độ dài (DP).

### Lỗi 6 (bonus): không cập nhật `mx` khi mở rộng đoạn

In [28]:
a, k = next((a, k) for a, k in pool if bug_stale_max(a, k) != max_sum_brute(a, k))
print("phản ví dụ:", a, k, "| bug:", bug_stale_max(a, k), "| đúng:", max_sum_brute(a, k))
assert bug_stale_max(a, k) != max_sum_brute(a, k)

phản ví dụ: [1, 0] 2 | bug: 1 | đúng: 2


**Vì sao sai:** khi đoạn mở rộng sang trái phải cập nhật `mx = max(mx, arr[i - L])`. Cố định `mx = arr[i-1]` bỏ qua phần tử lớn hơn nằm trong đoạn.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chia mảng thành các đoạn liên tiếp với giới hạn độ dài, giá trị mỗi đoạn là hàm của đoạn, tối ưu tổng" → **DP chia đoạn**: `f[i] = best_{j} f[j] + cost(j, i)`.

**Mẹo:**

1. Chốt **đoạn cuối** rồi đệ quy xuống tiền tố (hoặc **đoạn đầu** và hậu tố).
2. Duyệt độ dài lùi từ `i` và cập nhật `mx` `O(1)` nên mỗi `i` tốn `O(k)`.
3. `f[0] = 0` (mảng rỗng) là cơ sở.
4. Ghi `choice[i]` để khôi phục cách chia.
5. Oracle: vét cạn mặt nạ cắt `2^(n-1)`, nhanh cho `n ≤ 14`.
6. Kiểm bằng tính chất: biên `sum ≤ ans ≤ n·max`, đơn điệu theo `k`, dịch `+c ⇒ +n·c`.

**Họ bài cùng pattern:** Partition Array Into Disjoint Intervals (915), Palindrome Partitioning II (132), Word Break (139), Minimum Cost to Cut a Stick (1547), Split Array Largest Sum (410), Maximum Sum of 3 Non-Overlapping Subarrays (689).

## 15. Biến thể

1. **Khôi phục cách chia** tối ưu (đã làm ở mục 8): `best_partition`.
2. **Đếm số cách chia tối ưu:** DP đếm song song với `f`.
3. **Thay `max` bằng `min`:** mỗi đoạn đổi thành `min` của đoạn; tối đa hoá tổng (cùng DP, chỉ đổi `max` trong cửa sổ thành `min`).
4. **Chia thành đúng `m` đoạn** (mỗi đoạn dài `≤ k`): DP hai chiều `f[j][i]`.

Cài đặt và đối chiếu vét cạn:

In [29]:
def count_optimal(arr, k):
    n = len(arr)
    f = [0] * (n + 1)
    c = [0] * (n + 1)
    c[0] = 1
    for i in range(1, n + 1):
        mx = 0
        f[i] = -1
        for L in range(1, min(k, i) + 1):
            mx = max(mx, arr[i - L])
            v = f[i - L] + L * mx
            if v > f[i]:
                f[i], c[i] = v, c[i - L]
            elif v == f[i]:
                c[i] += c[i - L]
    return f[n], c[n]


def max_sum_min_variant(arr, k):
    n = len(arr)
    f = [0] * (n + 1)
    for i in range(1, n + 1):
        mn = 10 ** 18
        for L in range(1, min(k, i) + 1):
            mn = min(mn, arr[i - L])
            f[i] = max(f[i], f[i - L] + L * mn)
    return f[n]


def max_sum_exact_pieces(arr, k, m):
    """Chia thành đúng m đoạn (mỗi đoạn dài <= k); trả về None nếu không thể."""
    n = len(arr)
    NEG = None
    f = [[NEG] * (n + 1) for _ in range(m + 1)]
    f[0][0] = 0
    for j in range(1, m + 1):
        for i in range(1, n + 1):
            mx = 0
            for L in range(1, min(k, i) + 1):
                mx = max(mx, arr[i - L])
                if f[j - 1][i - L] is not None:
                    v = f[j - 1][i - L] + L * mx
                    if f[j][i] is None or v > f[j][i]:
                        f[j][i] = v
    return f[m][n]


def brute_variants(arr, k):
    n = len(arr)
    cnt_best, best, mins, exact = 0, 0, 0, {}
    for mask in range(1 << (n - 1)):
        parts, start = [], 0
        for j in range(n):
            if j == n - 1 or (mask >> j) & 1:
                parts.append(arr[start:j + 1])
                start = j + 1
        if max(len(p) for p in parts) > k:
            continue
        v = sum(len(p) * max(p) for p in parts)
        if v > best:
            best, cnt_best = v, 1
        elif v == best:
            cnt_best += 1
        mins = max(mins, sum(len(p) * min(p) for p in parts))
        exact[len(parts)] = max(exact.get(len(parts), -1), v)
    return best, cnt_best, mins, exact


rng = random.Random(15)
for _ in range(400):
    n = rng.randint(1, 10)
    a = [rng.randint(0, 6) for _ in range(n)]
    k = rng.randint(1, n)
    best, cnt_best, mins, exact = brute_variants(a, k)
    assert count_optimal(a, k) == (best, cnt_best)
    assert max_sum_min_variant(a, k) == mins
    for m in range(1, n + 1):
        assert max_sum_exact_pieces(a, k, m) == exact.get(m)
print("khôi phục / đếm số cách tối ưu / biến thể min / đúng m đoạn: khớp vét cạn")

khôi phục / đếm số cách tối ưu / biến thể min / đúng m đoạn: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | `2^(n-1)` cách cắt | `O(2^n · n)` | `O(1)` | oracle `n ≤ 14` |
| 2. Top-down | `go(i)` hậu tố có nhớ | `O(nk)` | `O(n)` | dễ viết từ công thức |
| 3. Prefix DP | `f[i]` từ `f[i-L]` | `O(nk)` | `O(n)` | **đáp án chính** |
| 4. Suffix DP | `g[i]` từ `g[i+L]` | `O(nk)` | `O(n)` | đối xứng gương |
| 5. Push DP | đẩy `f[i] → f[i+L]` | `O(nk)` | `O(n)` | cùng đồ thị trạng thái |

Mọi cách DP khớp vét cạn trên toàn bộ mảng nhỏ (`n ≤ 7`) và `500` mảng ngẫu nhiên `n ≤ 14`; khớp nhau trên `n = 500`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().maxSumAfterPartitioning([1, 2, 3, 4], 2), Solution().maxSumAfterPartitioning([4, 3, 2, 1], 4), Solution().maxSumAfterPartitioning([1, 9, 1], 2), Solution().maxSumAfterPartitioning([6, 6, 6], 1))

12 16 19 18


### Bài 2 (tính tay)

`arr = [1, 9, 1]`, `k = 2`. Lập bảng `f[0..3]` bằng tay (ghi các lựa chọn `(L, mx, giá trị)`). Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP tiền tố:

In [31]:
def max_sum_todo(arr, k):
    n = len(arr)
    f = [0] * (n + 1)
    for i in range(1, n + 1):
        mx = 0
        for L in range(1, min(k, i) + 1):
            pass        # TODO 1: cập nhật mx với arr[i - L]
            pass        # TODO 2: f[i] = max(f[i], f[i - L] + L * mx)
    return f[n]


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(max_sum_todo(a, k) == w for a, k, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **DP hậu tố** hoặc **đệ quy có nhớ**:

In [32]:
class MySolution:
    def maxSumAfterPartitioning(self, arr, k):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().maxSumAfterPartitioning(a, k) == w for a, k, w in examples)

### Bài 5 (mini challenge): chia thành đúng `m` đoạn

Viết `exact_pieces(arr, k, m)` trả về tổng lớn nhất khi chia thành **đúng** `m` đoạn (mỗi đoạn dài `≤ k`), hoặc `None` nếu không thể. Gợi ý: thêm chiều số đoạn vào trạng thái, `f[j][i]`.

In [33]:
def exact_pieces(arr, k, m):
    pass            # TODO


# Khi làm xong:
# assert exact_pieces([1, 15, 7, 9, 2, 5, 10], 3, 3) == 84
# assert exact_pieces([1, 15, 7, 9, 2, 5, 10], 3, 2) is None

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: tính từng bài bằng DP `f[i]`.
- Mức 2: `[4,3,2,1]` với `k = 4` ⇒ `4 · 4`; `k = 1` ⇒ tổng.
- Mức 3: `[1,2,3,4], k=2` có `f = 0, 1, 4, 7, 12`; kết quả bốn bài: `12, 16, 19, 18`.

**Bài 2**

- Mức 1: `f[0] = 0`, `f[1] = 1`.
- Mức 2: `f[2]`: `(1, 9, 1 + 9 = 10)`, `(2, 9, 0 + 18 = 18)` ⇒ `18`.
- Mức 3: `f[3]`: `(1, 1, 18 + 1 = 19)`, `(2, 9, 1 + 18 = 19)` ⇒ `19`.

**Bài 3**

- Mức 1: `mx = max(mx, arr[i - L])`.
- Mức 2: `f[i] = max(f[i], f[i - L] + L * mx)`.
- Mức 3: `range(1, min(k, i) + 1)`; trả `f[n]`.

**Bài 4**

- Mức 1: `g[n] = 0`; duyệt `i` giảm dần.
- Mức 2: thử độ dài đoạn đầu `L`, giữ `mx` của `arr[i..i+L-1]`.
- Mức 3: `g[i] = max(g[i], L * mx + g[i + L])`; trả `g[0]`.

**Bài 5**

- Mức 1: `f[j][i]` = tốt nhất khi chia `i` phần tử đầu thành đúng `j` đoạn.
- Mức 2: `f[0][0] = 0`, các ô khác `None` (không khả thi).
- Mức 3: `f[j][i] = max(f[j-1][i-L] + L·mx)` chỉ khi `f[j-1][i-L]` khả thi; trả `f[m][n]`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = (Solution().maxSumAfterPartitioning([1, 2, 3, 4], 2), Solution().maxSumAfterPartitioning([4, 3, 2, 1], 4), Solution().maxSumAfterPartitioning([1, 9, 1], 2), Solution().maxSumAfterPartitioning([6, 6, 6], 1))
print(vals)
assert vals == (max_sum_brute([1, 2, 3, 4], 2), max_sum_brute([4, 3, 2, 1], 4), max_sum_brute([1, 9, 1], 2), 18)
assert vals == (12, 16, 19, 18)
print("Bài 1 OK")

(12, 16, 19, 18)
Bài 1 OK


### Bài 2

In [35]:
f = trace_prefix([1, 9, 1], 2)
assert f == [0, 1, 18, 19]
assert max_sum_brute([1, 9, 1], 2) == 19
print("Bài 2 OK: f =", f)

arr = [1, 9, 1], k = 2
  i = 1: đoạn cuối arr[1-L..0], các lựa chọn (L, mx, giá trị) = [(1, 1, 1)] -> f[1] = 1
  i = 2: đoạn cuối arr[2-L..1], các lựa chọn (L, mx, giá trị) = [(1, 9, 10), (2, 9, 18)] -> f[2] = 18
  i = 3: đoạn cuối arr[3-L..2], các lựa chọn (L, mx, giá trị) = [(1, 1, 19), (2, 9, 19)] -> f[3] = 19
Bài 2 OK: f = [0, 1, 18, 19]


### Bài 3

In [36]:
def max_sum_answer(arr, k):
    n = len(arr)
    f = [0] * (n + 1)
    for i in range(1, n + 1):
        mx = 0
        for L in range(1, min(k, i) + 1):
            mx = max(mx, arr[i - L])
            f[i] = max(f[i], f[i - L] + L * mx)
    return f[n]


assert all(max_sum_answer(a, k) == w for a, k, w in examples)
for a, k in pool[:2000]:
    assert max_sum_answer(a, k) == max_sum_brute(a, k)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def maxSumAfterPartitioning(self, arr, k):
        n = len(arr)

        @lru_cache(maxsize=None)
        def go(i):
            if i == n:
                return 0
            best, mx = 0, 0
            for L in range(1, min(k, n - i) + 1):
                mx = max(mx, arr[i + L - 1])
                best = max(best, L * mx + go(i + L))
            return best

        return go(0)


assert all(MySolutionAnswer().maxSumAfterPartitioning(a, k) == w for a, k, w in examples)
for a, k in pool[:2000]:
    assert MySolutionAnswer().maxSumAfterPartitioning(a, k) == max_sum_brute(a, k)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def exact_pieces(arr, k, m):
    n = len(arr)
    f = [[None] * (n + 1) for _ in range(m + 1)]
    f[0][0] = 0
    for j in range(1, m + 1):
        for i in range(1, n + 1):
            mx = 0
            for L in range(1, min(k, i) + 1):
                mx = max(mx, arr[i - L])
                if f[j - 1][i - L] is not None:
                    v = f[j - 1][i - L] + L * mx
                    if f[j][i] is None or v > f[j][i]:
                        f[j][i] = v
    return f[m][n]


assert exact_pieces([1, 15, 7, 9, 2, 5, 10], 3, 3) == 84
assert exact_pieces([1, 15, 7, 9, 2, 5, 10], 3, 2) is None
rng = random.Random(19)
for _ in range(200):
    n = rng.randint(1, 9)
    a = [rng.randint(0, 6) for _ in range(n)]
    k = rng.randint(1, n)
    _, _, _, exact = brute_variants(a, k)
    for m in range(1, n + 1):
        assert exact_pieces(a, k, m) == exact.get(m)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (DP chia đoạn liên tiếp, độ dài ≤ k, giá trị đoạn phụ thuộc max):**

```python
class Solution:
    def maxSumAfterPartitioning(self, arr: List[int], k: int) -> int:
        n = len(arr)
        f = [0] * (n + 1)
        for i in range(1, n + 1):
            mx = 0
            for L in range(1, min(k, i) + 1):
                mx = max(mx, arr[i - L])
                f[i] = max(f[i], f[i - L] + L * mx)
        return f[n]
```

**Checklist trước khi submit:**

- [ ] `f[0] = 0`, đáp án `f[n]`?
- [ ] Độ dài đoạn cuối `L` chạy `1..min(k, i)` (có `+ 1` ở `range`)?
- [ ] `mx` cập nhật **theo đoạn** `arr[i-L..i-1]`, không phải toàn tiền tố?
- [ ] Giá trị đoạn là `L * mx` (đúng `L` phần tử)?
- [ ] Đã thử `n = 1`, `k = 1` (tổng), `k = n` (`n · max`), mảng toàn `0`, số `10^9`?
- [ ] Đúng tên `class Solution`, `maxSumAfterPartitioning(self, arr, k)`, không in debug.